We will add some new properties to the `entity` object to allow it to move about a plane to find food. With this addition, we can simulate a population growth contrained by limited available food resources. 

In [3]:
# Import required modules
import numpy as np
import matplotlib.pyplot as plt
from fractions import Fraction
from random import random, randint
from collections import Counter, namedtuple
from math import hypot
from abc import ABC, abstractmethod

Collision detection:

2 objects
https://www.youtube.com/watch?v=XYzA_kPWyJ8
multiple objects
https://www.youtube.com/watch?v=789weryntzM


In [4]:
class Vector:
    """ A vector stores information and operations regarding the 
        direction and speed of motion.
    
        Parameters:
            vx (int or float): x component of Vector.
            vy (int or float): y component of Vector.
    """

    def __init__(self, vx, vy):
        self.vx = vx
        self.vy = vy      

    def __repr__(self):
        return f"Vector({self.vx!r}, {self.vy!r})"

    def __bool__(self):
        """ Check truthiness of vector."""
        return bool(abs(self))

    def __add__(self, other):
        """ Add two vectors."""
        vx = self.vx + other.vx
        vy = self.vy + other.vy
        return Vector(vx, vy)

    def __mul__(self, scalar):
        """ Scale the vector."""
        return Vector(self.vx * scalar, self.vy * scalar)

    def mag(self):
        """ Get the magnitude of the vector."""
        return hypot(self.vx, self.vy)

    def set_mag(self, mag):
        """ Set the vector to the specified magnitude, maintaining its direction."""
        scalar = mag / self.mag()
        self.vx = self.vx * scalar
        self.vy = self.vy * scalar

    def unit(self):
        """ Get the unit vector of the vector."""
        mag = self.mag()
        return Vector(self.vx / mag, self.vy / mag)
    
    def set_unit(self):
        """ Set vector to its unit vector."""
        mag = self.mag()
        self.vx = self.vx / mag
        self.vy = self.vy / mag
    
    @staticmethod
    def generate(mag):
        vector = Vector(random() - 0.5, random() - 0.5)
        vector.set_mag(mag)
        return vector
        
        

In [5]:
Extent = namedtuple("area", ["width", "height"])

def trial(chance):
    """ Returns the boolean outcome of a trial given it's chance of success."""
    return random() <= chance

def distance(source, other):
    """ Returns the distance between two Entity objects."""
    dx = source.x - other.x
    dy = source.y - other.y
    return hypot(dx, dy)


def collision(source, other):
    """ Detect whether two round objects are touching."""
    if distance(source, other) <= source.radius + other.radius:
        return True
    return False

class Entity(ABC):
    """ Abstract base class for entities. An entity is any interactable object in 
        the Environment that has location (x, y).
    """

    def __init__(self, x, y):
        self.x = x
        self.y = y
    
    def __iadd__(self, other):
        """  Update the location of self by Vector other."""
        self.x = self.x + other.vx
        self.y = self.y + other.vy
        return self
    
    @staticmethod
    @abstractmethod
    def spawn():
        """ Every child of Entity must have a static spawn method."""
        pass

class Organism(Entity):
    """ The Organism object represents a live entity.

        Parameters:
            x (int or float): x location of Organism.
            y (int or float): y location of Organism.
            radius (float):   The radius of the Organism.
            motion (Vector):  The Organism's speed and direction.
    """
    
    def __init__(self, x, y, radius, motion):
        super().__init__(x, y)
        self.radius = radius
        self.motion = motion
        self.period = 0
    
    def __repr__(self):
        return f"Organism< age: {self.period!r}, location({self.x!r}, {self.y!r})>"
    
    def __call__(self):
        """ advance the entity by one period."""
        self.period += 1
        return self

    def spawn(environment):
        return Organism(
            randint(0, environment.extent.width),
            randint(0, environment.extent.height),
            1,
            Vector.generate(.2)
        )


class Food(Entity):
    """ The Food object provides energy for a live
        entity. This food object is stationary.

        Parameters:
            x (int or float): x location of Food.
            y (int or float): y location of Food.
            energy (float):   The food's energy value.
    """

    def __init__(self, x, y, energy = 1):
        super().__init__(x, y)
        self.energy = energy
    
    def __repr__(self):
        return f""

    def spawn(environment):
        return Food(
            randint(0, environment.extent.width),
            randint(0, environment.extent.height),
        )

class Environment():
    """ The environment tracks the population and parameters of the simulation.

        Parameters:
            repro_chance (float):               Chance of reproduction
            death_chance (float):               Chance of death
            starting_pop (int):                 Starting population
            extent (iterable [width, height]):  Extent of habitat
    """

    def __init__(self, repro_chance, death_chance, starting_pop, extent):
        self.repro_chance = repro_chance
        self.death_chance = death_chance
        self.starting_pop = starting_pop
        self.extent = Extent(*extent)

        self.population = [Organism.spawn(self) for _ in range(0, starting_pop)]
        self.history = [starting_pop]
        self.mortuary = Counter()

    def __repr__(self):
        return f"Environment <extent: {self.extent!r}, population: {self.population!r}>"

    def __call__(self):
        """ Advance the simulation by one period."""
        # Simulate period
        survive, expire, births = [], [], []
        # First determine which entities expired by end of previous period.
        for e in self.population:
            if trial(self.death_chance):
               expire.append(e().period) 
            else:
                survive.append(e())
                # Determine if the entity reproduced by end of current period.
                if trial(self.repro_chance):
                    births.append(Organism.spawn(self))

        # Finally, update attributes
        self.population = survive + births
        self.mortuary.update(expire)
        self.history.append(len(self.population))



In [6]:
e1 = Entity(3, 3, .5, Vector(.5, .5))
e2 = Entity(6, 6, .5, Vector(-.5, -.5))


In [12]:

e1()
print(e1, e1.x, e1.y)
e2()
print(e2, e2.x, e2.y)
print(collision(e1, e2))

6 6.0 6.0
6 3.0 3.0
False
